In [2]:
import pandas as pd
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

df = pd.read_csv('tourism_cleaned_engineered.csv')

# User-Item matrix: rows=users, columns=attractions, values=rating
user_item_matrix = df.pivot_table(index='UserId', columns='AttractionId', values='Rating')
print("Matrix shape:", user_item_matrix.shape)
print("Sparsity:", (user_item_matrix.isnull().sum().sum() / user_item_matrix.size * 100).round(1), "% empty")

Matrix shape: (33530, 30)
Sparsity: 95.5 % empty


In [3]:
item_user_matrix = user_item_matrix.fillna(0).T  # transpose: ab rows=attractions, cols=users

item_similarity = cosine_similarity(item_user_matrix)
item_similarity_df = pd.DataFrame(item_similarity, index=item_user_matrix.index, columns=item_user_matrix.index)

print("Item similarity matrix shape:", item_similarity_df.shape)

# Test —  popular attraction (640)  similar attractions 
print("\nMost similar to attraction 640:")
print(item_similarity_df[640].sort_values(ascending=False).head(6))

Item similarity matrix shape: (30, 30)

Most similar to attraction 640:
AttractionId
640    1.000000
748    0.274019
749    0.174610
737    0.116497
841    0.110575
824    0.104087
Name: 640, dtype: float64


In [4]:
item = pd.read_excel(
    'Tourism_Analytics/data/Additional_Data_for_Attraction_Sites/Updated_Item.xlsx'
)
attraction_names = item.set_index('AttractionId')['Attraction']

top_similar = item_similarity_df[640].sort_values(ascending=False).head(6)
for aid, score in top_similar.items():
    print(f"{attraction_names.get(aid, 'Unknown')}: {score:.3f}")

Sacred Monkey Forest Sanctuary: 1.000
Tegalalang Rice Terrace: 0.274
Tegenungan Waterfall: 0.175
Tanah Lot Temple: 0.116
Waterbom Bali: 0.111
Uluwatu Temple: 0.104


In [5]:
def recommend_similar_attractions(attraction_id, top_n=5):
    if attraction_id not in item_similarity_df.columns:
        return "Attraction not in transaction history — use content-based instead."
    similar_scores = item_similarity_df[attraction_id].sort_values(ascending=False)
    similar_scores = similar_scores.drop(attraction_id)  # khud ko hata do
    top_ids = similar_scores.head(top_n).index
    result = pd.DataFrame({
        'AttractionId': top_ids,
        'Attraction': [attraction_names.get(i, 'Unknown') for i in top_ids],
        'SimilarityScore': similar_scores.head(top_n).values
    })
    return result

print(recommend_similar_attractions(640))

   AttractionId               Attraction  SimilarityScore
0           748  Tegalalang Rice Terrace         0.274019
1           749     Tegenungan Waterfall         0.174610
2           737         Tanah Lot Temple         0.116497
3           841            Waterbom Bali         0.110575
4           824           Uluwatu Temple         0.104087


In [6]:
item = pd.read_excel(
    'Tourism_Analytics/data/Additional_Data_for_Attraction_Sites/Updated_Item.xlsx'
)

type_df = pd.read_excel('Tourism_Analytics/data/Type.xlsx')

item_full = item.merge(type_df, on='AttractionTypeId', how='left')
# Content-based: same type + same city = similar. One-hot encode dono
content_features = pd.get_dummies(item_full[['AttractionTypeId', 'AttractionCityId']].astype(str))
content_similarity = cosine_similarity(content_features)
content_similarity_df = pd.DataFrame(content_similarity, index=item_full['AttractionId'], columns=item_full['AttractionId'])

def recommend_content_based(attraction_id, top_n=5):
    if attraction_id not in content_similarity_df.columns:
        return "Attraction ID not found."
    scores = content_similarity_df[attraction_id].sort_values(ascending=False).drop(attraction_id)
    top_ids = scores.head(top_n).index
    result = pd.DataFrame({
        'AttractionId': top_ids,
        'Attraction': item_full.set_index('AttractionId').loc[top_ids, 'Attraction'].values,
        'SimilarityScore': scores.head(top_n).values
    })
    return result

print(recommend_content_based(640))

# Ab ek attraction pe test karो jo Transaction data mein NAHI thi (proving 1,698-catalog extension kaam karta hai)
all_ids = item_full['AttractionId'].unique()
transaction_ids = set(item_similarity_df.columns)
never_rated = [i for i in all_ids if i not in transaction_ids][:1]
print(f"\nContent-based recommendation for never-rated attraction {never_rated[0]}:")
print(recommend_content_based(never_rated[0]))

   AttractionId               Attraction  SimilarityScore
0          1308  Paradise Beach - Douala              0.5
1          1309   Sacred Shrine - Douala              0.5
2          1306        Eco Park - Douala              0.5
3          1307     Art Gallery - Douala              0.5
4           975             Sempu Island              0.5

Content-based recommendation for never-rated attraction 1298:
   AttractionId                          Attraction  SimilarityScore
0          1455              Sacred Shrine - Luanda              0.5
1          1965                 Cathedral - Banqiao              0.5
2          1457               Grand Mosque - Luanda              0.5
3          1458                Cathedral - Gaborone              0.5
4          2074  Sacred Shrine - Muscat Governorate              0.5


In [7]:
type_dummies = pd.get_dummies(item_full['AttractionTypeId'].astype(str))
city_dummies = pd.get_dummies(item_full['AttractionCityId'].astype(str))

type_sim = cosine_similarity(type_dummies)
city_sim = cosine_similarity(city_dummies)

combined_sim = 0.3 * type_sim + 0.7 * city_sim

content_similarity_df = pd.DataFrame(combined_sim, index=item_full['AttractionId'], columns=item_full['AttractionId'])

# Dobara test karo
print(recommend_content_based(640))
print(f"\nContent-based recommendation for never-rated attraction {never_rated[0]}:")
print(recommend_content_based(never_rated[0]))

   AttractionId               Attraction  SimilarityScore
0          1308  Paradise Beach - Douala              0.7
1          1309   Sacred Shrine - Douala              0.7
2          1306        Eco Park - Douala              0.7
3          1307     Art Gallery - Douala              0.7
4           650              Sanur Beach              0.7

Content-based recommendation for never-rated attraction 1298:
   AttractionId                    Attraction  SimilarityScore
0          1300             National Park - -              0.7
1          1301  Cultural Heritage Center - -              0.7
2          1299            Paradise Beach - -              0.7
3          1455        Sacred Shrine - Luanda              0.3
4          1965           Cathedral - Banqiao              0.3


In [8]:
print("Attraction 640 (Bali):")
print(item_full[item_full['AttractionId']==640][['AttractionId','Attraction','AttractionCityId']])

print("\nDouala attractions:")
print(item_full[item_full['Attraction'].str.contains('Douala')][['AttractionId','Attraction','AttractionCityId']])

print("\nHow many attractions have missing AttractionCityId:")
print(item_full['AttractionCityId'].isnull().sum())

Attraction 640 (Bali):
   AttractionId                      Attraction  AttractionCityId
2           640  Sacred Monkey Forest Sanctuary                 1

Douala attractions:
    AttractionId               Attraction  AttractionCityId
38          1306        Eco Park - Douala                 1
39          1307     Art Gallery - Douala                 1
40          1308  Paradise Beach - Douala                 1
41          1309   Sacred Shrine - Douala                 1

How many attractions have missing AttractionCityId:
0


In [9]:
# Kitne attractions CityId=1 share karte hain?
print("Attractions with AttractionCityId=1:", (item_full['AttractionCityId']==1).sum())
print(item_full[item_full['AttractionCityId']==1][['AttractionId','Attraction']].head(10))

# City.xlsx mein CityId=1 actually kya hai?
city = pd.read_excel('Tourism_Analytics/data/City.xlsx')
print("\nCityId=1 in City.xlsx:", city[city['CityId']==1])

Attractions with AttractionCityId=1: 14
   AttractionId                      Attraction
0           369               Kuta Beach - Bali
1           481                  Nusa Dua Beach
2           640  Sacred Monkey Forest Sanctuary
3           650                     Sanur Beach
4           673                  Seminyak Beach
5           737                Tanah Lot Temple
6           748         Tegalalang Rice Terrace
7           749            Tegenungan Waterfall
8           824                  Uluwatu Temple
9           841                   Waterbom Bali

CityId=1 in City.xlsx:    CityId CityName  CountryId
1       1   Douala          1


In [10]:
has_dash = item_full['Attraction'].str.contains(' - ', regex=False)
print("Attractions with ' - City' pattern in name:", has_dash.sum(), "out of", len(item_full))
print("\nSample without dash pattern:")
print(item_full[~has_dash]['Attraction'].head(10).tolist())

Attractions with ' - City' pattern in name: 1669 out of 1698

Sample without dash pattern:
['Nusa Dua Beach', 'Sacred Monkey Forest Sanctuary', 'Sanur Beach', 'Seminyak Beach', 'Tanah Lot Temple', 'Tegalalang Rice Terrace', 'Tegenungan Waterfall', 'Uluwatu Temple', 'Waterbom Bali', 'Balekambang Beach']


In [11]:
item_full['city_from_name'] = item_full['Attraction'].apply(
    lambda x: x.split(' - ')[-1].strip() if ' - ' in x else 'Unknown'
)

print("Unique cities extracted:", item_full['city_from_name'].nunique())
print(item_full['city_from_name'].value_counts().head(10))

# Content-based similarity — ab sahi city + type dono se
type_dummies = pd.get_dummies(item_full['AttractionTypeId'].astype(str))
city_dummies = pd.get_dummies(item_full['city_from_name'])

type_sim = cosine_similarity(type_dummies)
city_sim = cosine_similarity(city_dummies)
combined_sim = 0.3 * type_sim + 0.7 * city_sim

content_similarity_df = pd.DataFrame(combined_sim, index=item_full['AttractionId'], columns=item_full['AttractionId'])

# Dobara test — 640 (Bali) pe
print("\n640 (Bali) recommendations:")
print(recommend_content_based(640))

print(f"\nNever-rated attraction {never_rated[0]}:")
print(recommend_content_based(never_rated[0]))

Unique cities extracted: 418
city_from_name
Unknown            29
Tripoli             8
-                   4
South Region        4
Douala              4
N'Djamena           4
Kigali Province     4
Kigali              4
Addis Ababa         4
Karen               4
Name: count, dtype: int64

640 (Bali) recommendations:
   AttractionId                    Attraction  SimilarityScore
0           975                  Sempu Island              1.0
1          1220  Ramayana Ballet at Prambanan              0.7
2          1225              Ratu Boko Temple              0.7
3          1238                   Sewu Temple              0.7
4          1278          Ullen Sentalu Museum              0.7

Never-rated attraction 1298:
   AttractionId                    Attraction  SimilarityScore
0          1300             National Park - -              0.7
1          1301  Cultural Heritage Center - -              0.7
2          1299            Paradise Beach - -              0.7
3          1455      

In [12]:
# "Unknown" aur "-" dono ko missing treat karo
item_full['city_from_name'] = item_full['city_from_name'].replace({'Unknown': None, '-': None})
item_full['city_from_name'] = item_full['city_from_name'].str.strip().replace('', None)

# dummy_na=False (default) ke saath, NaN wale rows ki dummy row automatically all-zero ban jaएगी
# matlab unकी city-similarity kisी se bhी (khud jaisो se bhी) match nahi hogी — sirf type match count hoga
city_dummies = pd.get_dummies(item_full['city_from_name'])  # NaN wale apne aap zero-row ban jaएnge
type_dummies = pd.get_dummies(item_full['AttractionTypeId'].astype(str))

type_sim = cosine_similarity(type_dummies)
city_sim = cosine_similarity(city_dummies)
combined_sim = 0.3 * type_sim + 0.7 * city_sim

content_similarity_df = pd.DataFrame(combined_sim, index=item_full['AttractionId'], columns=item_full['AttractionId'])

print("640 (Bali) recommendations:")
print(recommend_content_based(640))

print(f"\nNever-rated attraction {never_rated[0]}:")
print(recommend_content_based(never_rated[0]))

640 (Bali) recommendations:
   AttractionId                               Attraction  SimilarityScore
0           975                             Sempu Island              0.3
1          2949             Farmers' Market - Roosendaal              0.0
2          2934                     Art Gallery - Monaco              0.0
3          2951           Farmers' Market - Isle of Mull              0.0
4          2952  Cultural Heritage Center - Isle of Mull              0.0

Never-rated attraction 1298:
   AttractionId              Attraction  SimilarityScore
0          1811    Cathedral - Valdivia              0.3
1          1453   Sacred Shrine - Tunis              0.3
2          1454   Grand Mosque - Luanda              0.3
3          1455  Sacred Shrine - Luanda              0.3
4          1965     Cathedral - Banqiao              0.3


In [13]:
print("Attraction 640's type:", item_full[item_full['AttractionId']==640]['AttractionTypeId'].values)
print("\nHow many attractions share that same AttractionTypeId?")
target_type = item_full[item_full['AttractionId']==640]['AttractionTypeId'].values[0]
print((item_full['AttractionTypeId'] == target_type).sum())

print("\nAttractionTypeId dtype:", item_full['AttractionTypeId'].dtype)
print("Sample values:", item_full['AttractionTypeId'].unique()[:10])

Attraction 640's type: [63]

How many attractions share that same AttractionTypeId?
2

AttractionTypeId dtype: object
Sample values: [13 63 76 72 93 92 61 64 82 91]


In [14]:
# Ek attraction dhoondo jiski city aur type dono valid hon, testing ke liye
clean_sample = item_full[item_full['city_from_name'].notna()]['AttractionId'].iloc[10]
print(f"Testing with clean attraction {clean_sample}: {item_full[item_full['AttractionId']==clean_sample]['Attraction'].values[0]}")
print(recommend_content_based(clean_sample))

Testing with clean attraction 1311: Farmers' Market - N'Djamena
   AttractionId                            Attraction  SimilarityScore
0          1313           Farmers' Market - N'Djamena              1.0
1          1312  Cultural Heritage Center - N'Djamena              0.7
2          1310               Crystal Bay - N'Djamena              0.7
3          2950            Flea Market - Isle of Mull              0.3
4          2949          Farmers' Market - Roosendaal              0.3


In [15]:
type_lookup = pd.read_excel('Tourism_Analytics/data/Type.xlsx')
print("Type.xlsx AttractionTypeId values:", sorted(type_lookup['AttractionTypeId'].unique()))

print("\nUpdated_Item.xlsx AttractionTypeId unique count:", item_full['AttractionTypeId'].nunique())
print("Sample values:", sorted(item_full['AttractionTypeId'].astype(str).unique())[:20])

# Kितने attractions ka type Type.xlsx mein exist karta hai?
valid_types = set(type_lookup['AttractionTypeId'].astype(str))
item_types = set(item_full['AttractionTypeId'].astype(str))
print("\nOverlap between Item's types and Type.xlsx's types:", len(valid_types & item_types), "out of", len(item_types), "unique item types")

Type.xlsx AttractionTypeId values: [np.int64(2), np.int64(10), np.int64(13), np.int64(19), np.int64(34), np.int64(44), np.int64(45), np.int64(61), np.int64(63), np.int64(64), np.int64(72), np.int64(76), np.int64(82), np.int64(84), np.int64(91), np.int64(92), np.int64(93)]

Updated_Item.xlsx AttractionTypeId unique count: 22
Sample values: ['10', '13', '19', '2', '34', '44', '45', '61', '63', '64', '72', '76', '82', '84', '91', '92', '93', 'Beach', 'Market', 'Museum']

Overlap between Item's types and Type.xlsx's types: 17 out of 22 unique item types


In [16]:
# Kya Transaction data wale 30 attractions is problem se affected hain?
transaction_attraction_ids = df['AttractionId'].unique() if 'df' in dir() else None
transaction = pd.read_excel('Tourism_Analytics/data/Transaction.xlsx')
core_30_ids = transaction['AttractionId'].unique()

core_items = item_full[item_full['AttractionId'].isin(core_30_ids)]
print("Core 30 attractions - AttractionTypeId values:")
print(core_items['AttractionTypeId'].tolist())

# Kितne text-wali (naam wali) rows hain poore catalog mein?
is_text_type = ~item_full['AttractionTypeId'].astype(str).str.match(r'^\d+$')
print(f"\nRows with text-based (non-numeric) type: {is_text_type.sum()} out of {len(item_full)}")
print(item_full[is_text_type][['AttractionId','Attraction','AttractionTypeId']].head(10))

Core 30 attractions - AttractionTypeId values:
[13, 13, 63, 13, 13, 76, 72, 93, 76, 92, 13, 61, 93, 13, 64, 82, 72, 91, 84, 63, 19, 61, 34, 91, 10, 2, 2, 45, 72, 44]

Rows with text-based (non-numeric) type: 1668 out of 1698
    AttractionId                      Attraction AttractionTypeId
30          1298              Ancient Temple - -           Temple
31          1299              Paradise Beach - -            Beach
32          1300               National Park - -             Park
33          1301    Cultural Heritage Center - -           Museum
34          1302      Art Gallery - South Region           Museum
35          1303  Farmers' Market - South Region           Market
36          1304     Night Market - South Region           Market
37          1305        Cathedral - South Region           Temple
38          1306               Eco Park - Douala             Park
39          1307            Art Gallery - Douala           Museum


In [17]:
type_lookup_map = type_lookup.set_index('AttractionTypeId')['AttractionType'].to_dict()

def normalize_type(val):
    val_str = str(val).strip()
    if val_str.isdigit():
        return type_lookup_map.get(int(val_str), 'Other')
    return val_str  # already text jaisa "Beach", "Market" — waisa hi rakho

item_full['AttractionType_clean'] = item_full['AttractionTypeId'].apply(normalize_type)

print(item_full['AttractionType_clean'].value_counts())

AttractionType_clean
Museum                            367
Temple                            340
Market                            325
Beach                             318
Park                              318
Beaches                             6
Points of Interest & Landmarks      3
Nature & Wildlife Areas             2
Religious Sites                     2
Waterfalls                          2
National Parks                      2
Volcanos                            2
Ancient Ruins                       2
Water Parks                         1
Neighborhoods                       1
Spas                                1
Speciality Museums                  1
Caverns & Caves                     1
Flea & Street Markets               1
Ballets                             1
History Museums                     1
Historic Sites                      1
Name: count, dtype: int64


In [18]:
type_dummies = pd.get_dummies(item_full['AttractionType_clean'])
city_dummies = pd.get_dummies(item_full['city_from_name'])

type_sim = cosine_similarity(type_dummies)
city_sim = cosine_similarity(city_dummies)
combined_sim = 0.3 * type_sim + 0.7 * city_sim

content_similarity_df = pd.DataFrame(combined_sim, index=item_full['AttractionId'], columns=item_full['AttractionId'])

# Final test — same doosra clean attraction
print(recommend_content_based(clean_sample))

   AttractionId                            Attraction  SimilarityScore
0          1313           Farmers' Market - N'Djamena              1.0
1          1312  Cultural Heritage Center - N'Djamena              0.7
2          1310               Crystal Bay - N'Djamena              0.7
3          2950            Flea Market - Isle of Mull              0.3
4          2949          Farmers' Market - Roosendaal              0.3


In [19]:
def recommend_for_user(user_id, top_n=5):
    # User ne kaunse attractions already visit/rate kiye
    user_history = df[df['UserId'] == user_id][['AttractionId', 'Rating']]

    if user_history.empty:
        return "User not found."

    # Highest-rated attractions ko preference ke liye use karenge
    user_history = user_history.sort_values('Rating', ascending=False)

    scores = {}

    for _, row in user_history.iterrows():
        attraction_id = row['AttractionId']
        rating = row['Rating']

        # Sirf un attractions se recommendations lo
        # jo content-based catalog mein available hain
        if attraction_id not in content_similarity_df.columns:
            continue

        similar = content_similarity_df[attraction_id].drop(attraction_id)

        for candidate_id, similarity in similar.items():

            # User already visited/rated attraction ko recommend nahi karna
            if candidate_id in user_history['AttractionId'].values:
                continue

            # Rating × similarity = personalized score
            score = rating * similarity

            if candidate_id not in scores:
                scores[candidate_id] = 0

            scores[candidate_id] += score

    if not scores:
        return "No recommendations available."

    # Highest score first
    ranked = sorted(
        scores.items(),
        key=lambda x: x[1],
        reverse=True
    )[:top_n]

    result = pd.DataFrame(ranked, columns=['AttractionId', 'RecommendationScore'])

    result['Attraction'] = result['AttractionId'].map(
        item_full.set_index('AttractionId')['Attraction']
    )

    result = result[
        ['AttractionId', 'Attraction', 'RecommendationScore']
    ]

    return result

In [20]:
# Test with a user who has visited multiple attractions

test_user = df['UserId'].value_counts().index[0]

print("Test User:", test_user)

print("\nUser history:")
print(
    df[df['UserId'] == test_user][
        ['AttractionId', 'Attraction', 'Rating']
    ].sort_values('Rating', ascending=False)
)

print("\nPersonalized Recommendations:")
print(recommend_for_user(test_user, top_n=5))

Test User: 60799

User history:
       AttractionId                      Attraction  Rating
48965          1171                  Merapi Volcano       5
10049           640  Sacred Monkey Forest Sanctuary       4
4747            640  Sacred Monkey Forest Sanctuary       4
22128           673                  Seminyak Beach       4
22947           481                  Nusa Dua Beach       4
36405           748         Tegalalang Rice Terrace       4
33250           748         Tegalalang Rice Terrace       4
43184           369               Kuta Beach - Bali       4
48950          1171                  Merapi Volcano       4
48952          1171                  Merapi Volcano       4
20175           673                  Seminyak Beach       4
48966          1171                  Merapi Volcano       4
48964          1171                  Merapi Volcano       4
48958          1171                  Merapi Volcano       4
48953          1171                  Merapi Volcano       4
48961   

In [21]:
def hybrid_recommendation(user_id, top_n=5, alpha=0.5):
    """
    Hybrid Recommendation:
    alpha = weight for collaborative filtering
    (1-alpha) = weight for content-based filtering
    """

    # User history
    user_history = df[df['UserId'] == user_id][
        ['AttractionId', 'Rating']
    ].copy()

    if user_history.empty:
        return "User not found."

    visited_ids = set(user_history['AttractionId'])

    scores = {}

    # ---------------------------------
    # 1. CONTENT-BASED SCORE
    # ---------------------------------

    for _, row in user_history.iterrows():

        attraction_id = row['AttractionId']
        rating = row['Rating']

        if attraction_id not in content_similarity_df.columns:
            continue

        similarities = content_similarity_df[attraction_id]

        for candidate_id, similarity in similarities.items():

            if candidate_id in visited_ids:
                continue

            score = rating * similarity

            scores[candidate_id] = scores.get(candidate_id, 0) + score

    # ---------------------------------
    # 2. COLLABORATIVE SCORE
    # ---------------------------------

    for _, row in user_history.iterrows():

        attraction_id = row['AttractionId']
        rating = row['Rating']

        if attraction_id not in item_similarity_df.columns:
            continue

        similarities = item_similarity_df[attraction_id]

        for candidate_id, similarity in similarities.items():

            if candidate_id in visited_ids:
                continue

            score = rating * similarity

            scores[candidate_id] = scores.get(candidate_id, 0) + score

    # ---------------------------------
    # 3. CREATE RESULT
    # ---------------------------------

    if not scores:
        return "No recommendations available."

    result = pd.DataFrame(
        scores.items(),
        columns=['AttractionId', 'HybridScore']
    )

    result = result.sort_values(
        'HybridScore',
        ascending=False
    ).head(top_n)

    result['Attraction'] = result['AttractionId'].map(
        item_full.set_index('AttractionId')['Attraction']
    )

    return result[
        ['AttractionId', 'Attraction', 'HybridScore']
    ]

In [22]:
print("Hybrid Recommendations for User:", test_user)

print(
    hybrid_recommendation(
        test_user,
        top_n=5
    )
)

Hybrid Recommendations for User: 60799
    AttractionId            Attraction  HybridScore
11           947  Mount Semeru Volcano    51.775388
4            877     Balekambang Beach     6.730294
7            913        Goa Cina Beach     6.708308
2            824        Uluwatu Temple     6.325390
0            737      Tanah Lot Temple     6.239814


In [23]:
def hybrid_recommendation(user_id, top_n=5, alpha=0.5):

    user_history = df[df['UserId'] == user_id][
        ['AttractionId', 'Rating']
    ].copy()

    if user_history.empty:
        return "User not found."

    visited_ids = set(user_history['AttractionId'])

    content_scores = {}
    collaborative_scores = {}

    # -----------------------------
    # CONTENT-BASED
    # -----------------------------

    for _, row in user_history.iterrows():

        attraction_id = row['AttractionId']
        rating = row['Rating']

        if attraction_id not in content_similarity_df.columns:
            continue

        similarities = content_similarity_df[attraction_id]

        for candidate_id, similarity in similarities.items():

            if candidate_id in visited_ids:
                continue

            score = rating * similarity
            content_scores[candidate_id] = (
                content_scores.get(candidate_id, 0) + score
            )

    # -----------------------------
    # COLLABORATIVE
    # -----------------------------

    for _, row in user_history.iterrows():

        attraction_id = row['AttractionId']
        rating = row['Rating']

        if attraction_id not in item_similarity_df.columns:
            continue

        similarities = item_similarity_df[attraction_id]

        for candidate_id, similarity in similarities.items():

            if candidate_id in visited_ids:
                continue

            score = rating * similarity
            collaborative_scores[candidate_id] = (
                collaborative_scores.get(candidate_id, 0) + score
            )

    # -----------------------------
    # NORMALIZE SCORES
    # -----------------------------

    def normalize(scores):

        if not scores:
            return {}

        max_score = max(scores.values())

        if max_score == 0:
            return scores

        return {
            k: v / max_score
            for k, v in scores.items()
        }

    content_scores = normalize(content_scores)
    collaborative_scores = normalize(collaborative_scores)

    # -----------------------------
    # COMBINE
    # -----------------------------

    all_candidates = set(content_scores) | set(collaborative_scores)

    hybrid_scores = {}

    for candidate_id in all_candidates:

        content_score = content_scores.get(candidate_id, 0)
        collaborative_score = collaborative_scores.get(candidate_id, 0)

        hybrid_scores[candidate_id] = (
            alpha * collaborative_score
            + (1 - alpha) * content_score
        )

    # -----------------------------
    # FINAL RESULT
    # -----------------------------

    result = pd.DataFrame(
        hybrid_scores.items(),
        columns=['AttractionId', 'HybridScore']
    )

    result = result.sort_values(
        'HybridScore',
        ascending=False
    ).head(top_n)

    result['Attraction'] = result['AttractionId'].map(
        item_full.set_index('AttractionId')['Attraction']
    )

    return result[
        ['AttractionId', 'Attraction', 'HybridScore']
    ]

In [24]:
print("Final Hybrid Recommendations")
print("--------------------------------")

final_recommendations = hybrid_recommendation(
    test_user,
    top_n=5,
    alpha=0.5
)

print(final_recommendations)

Final Hybrid Recommendations
--------------------------------
    AttractionId                          Attraction  HybridScore
11           947                Mount Semeru Volcano     0.632434
2            824                      Uluwatu Temple     0.500000
0            737                    Tanah Lot Temple     0.493235
1            749                Tegenungan Waterfall     0.424795
5            888  Bromo Tengger Semeru National Park     0.346658


In [25]:
# Recommendation Validation

recommendations = hybrid_recommendation(
    test_user,
    top_n=5,
    alpha=0.5
)

print("Recommendation Validation")
print("--------------------------")

if isinstance(recommendations, str):
    print(recommendations)

else:
    recommended_ids = set(recommendations['AttractionId'])
    visited_ids = set(
        df[df['UserId'] == test_user]['AttractionId']
    )

    # Check 1: Already visited attractions
    repeated = recommended_ids.intersection(visited_ids)

    print("Test User:", test_user)
    print("Number of recommendations:", len(recommended_ids))
    print("Already visited recommendations:", len(repeated))

    # Check 2: Recommendation diversity
    unique_attractions = recommendations['Attraction'].nunique()

    print("Unique recommended attractions:", unique_attractions)

    # Check 3: Final recommendation list
    print("\nFinal Recommendations:")
    print(recommendations)

Recommendation Validation
--------------------------
Test User: 60799
Number of recommendations: 5
Already visited recommendations: 0
Unique recommended attractions: 5

Final Recommendations:
    AttractionId                          Attraction  HybridScore
11           947                Mount Semeru Volcano     0.632434
2            824                      Uluwatu Temple     0.500000
0            737                    Tanah Lot Temple     0.493235
1            749                Tegenungan Waterfall     0.424795
5            888  Bromo Tengger Semeru National Park     0.346658


In [26]:
# Recommendation Coverage

all_recommended = set()

sample_users = df['UserId'].drop_duplicates().head(100)

for user_id in sample_users:

    recs = hybrid_recommendation(
        user_id,
        top_n=5,
        alpha=0.5
    )

    if not isinstance(recs, str):
        all_recommended.update(
            recs['AttractionId'].tolist()
        )

total_attractions = item_full['AttractionId'].nunique()

coverage = (
    len(all_recommended) / total_attractions
) * 100

print("Recommendation Coverage")
print("------------------------")
print("Users tested:", len(sample_users))
print("Unique attractions recommended:", len(all_recommended))
print("Total attractions:", total_attractions)
print(f"Coverage: {coverage:.2f}%")

Recommendation Coverage
------------------------
Users tested: 100
Unique attractions recommended: 15
Total attractions: 1698
Coverage: 0.88%


In [27]:
# Final Recommendation Demo

demo_user = test_user

print("=" * 60)
print("TOURISM RECOMMENDATION SYSTEM")
print("=" * 60)

print(f"\nUser ID: {demo_user}")

print("\nPreviously Visited Attractions:")
print(
    df[df['UserId'] == demo_user][
        ['AttractionId', 'Attraction', 'Rating']
    ]
    .drop_duplicates()
    .sort_values('Rating', ascending=False)
    .head(10)
)

print("\nRecommended Attractions:")
print("-" * 60)

final_recommendations = hybrid_recommendation(
    demo_user,
    top_n=5,
    alpha=0.5
)

print(final_recommendations)

print("\nRecommendation Method:")
print("50% Collaborative Filtering + 50% Content-Based Filtering")

TOURISM RECOMMENDATION SYSTEM

User ID: 60799

Previously Visited Attractions:
       AttractionId                      Attraction  Rating
48965          1171                  Merapi Volcano       5
20175           673                  Seminyak Beach       4
4747            640  Sacred Monkey Forest Sanctuary       4
22947           481                  Nusa Dua Beach       4
43184           369               Kuta Beach - Bali       4
48950          1171                  Merapi Volcano       4
33250           748         Tegalalang Rice Terrace       4
24236           481                  Nusa Dua Beach       3
27237           650                     Sanur Beach       3
48951          1171                  Merapi Volcano       3

Recommended Attractions:
------------------------------------------------------------
    AttractionId                          Attraction  HybridScore
11           947                Mount Semeru Volcano     0.632434
2            824                      Uluw

## Recommendation System — Final Findings

The recommendation system combines Collaborative Filtering and
Content-Based Filtering to generate personalized tourism recommendations.

### Approach

1. Collaborative Filtering was used to identify attractions based on
   user-item interaction patterns.
2. Content-Based Filtering was used using cleaned attraction attributes,
   including Attraction Type and City.
3. A Hybrid Recommendation approach was developed by combining the
   collaborative and content-based scores.
4. Previously visited attractions were removed from the final
   recommendation list.

### Validation Results

For the test user:

- Number of recommendations: 5
- Already visited recommendations: 0
- Unique recommendations: 5

This confirms that the system can generate distinct recommendations
without recommending attractions already present in the user's history.

### Coverage

The system was tested on 100 users.

- Unique attractions recommended: 15
- Total attractions in catalog: 1,698
- Recommendation coverage: 0.88%

The relatively low coverage is a limitation of the current dataset and
interaction history. The transaction data contains interactions for a
much smaller subset of attractions compared with the complete attraction
catalog.

### Final Recommendation Method

The final hybrid system uses:

- 50% Collaborative Filtering
- 50% Content-Based Filtering

The system successfully demonstrates personalized tourism
recommendation generation while also highlighting the limitations of
the available interaction data.

In [28]:
import joblib
joblib.dump(content_similarity_df, 'content_similarity_matrix.pkl')
joblib.dump(item_full[['AttractionId', 'Attraction']], 'attraction_lookup.pkl')
print("Recommendation artifacts saved.")

Recommendation artifacts saved.
